In [1]:
!pip -q install huggingface-hub==1.31.0

In [2]:
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi

drive.mount("/content/drive")

MODEL_DIR = Path("/content/drive/MyDrive/distilbert_llm_router")

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Checkpoint folder not found: {MODEL_DIR}")

required = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

missing = [name for name in required if not (MODEL_DIR / name).exists()]

if missing:
    raise FileNotFoundError("Missing checkpoint files: " + ", ".join(missing))

print("Checkpoint folder:", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print(" -", p.name)

Mounted at /content/drive
Checkpoint folder: /content/drive/MyDrive/distilbert_llm_router
 - config.json
 - model.safetensors
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face write token "
        "to Colab Secrets using the key icon."
    )

api = HfApi(token=HF_TOKEN)
account = api.whoami()
username = account["name"]

print("Authenticated Hugging Face user:", username)

Authenticated Hugging Face user: Paam24


In [4]:
REPO_NAME = "cost-aware-llm-router-distilbert"
REPO_ID = f"{username}/{REPO_NAME}"

repo_url = api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

print("Model repository:", REPO_ID)
print("URL:", repo_url)

Model repository: Paam24/cost-aware-llm-router-distilbert
URL: https://huggingface.co/Paam24/cost-aware-llm-router-distilbert


In [5]:
api.upload_folder(
    folder_path=str(MODEL_DIR),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload trained DistilBERT router checkpoint",
)

print("Checkpoint upload complete.")

Checkpoint upload complete.


In [6]:
MODEL_CARD = """---
base_model: distilbert/distilbert-base-uncased
library_name: transformers
tags:
- llm-routing
- distilbert
- regression
- cost-aware-routing
- routerbench
---

# Cost-Aware LLM Router — DistilBERT Checkpoint

This repository contains the trained DistilBERT checkpoint used in the
Cost-Aware LLM Router college ML project.

## Task

Given a benchmark-formatted question, the model predicts 11 bounded performance
scores, one for each historical candidate LLM in the routing pool.

The outputs are regression scores, not calibrated probabilities.

## Base model

`distilbert-base-uncased`

## Training data

The training pipeline uses the public RouterBench 0-shot data and focuses on
MMLU, ARC-Challenge, and GSM8K / grade-school-math.

## Candidate model order

1. WizardLM/WizardLM-13B-V1.2
2. claude-instant-v1
3. claude-v1
4. claude-v2
5. gpt-3.5-turbo-1106
6. gpt-4-1106-preview
7. meta/code-llama-instruct-34b-chat
8. meta/llama-2-70b-chat
9. mistralai/mistral-7b-chat
10. mistralai/mixtral-8x7b-chat
11. zero-one-ai/Yi-34B-Chat

## Main training configuration

- maximum sequence length: 384
- learning rate: 2e-5
- training batch size: 16 per device
- evaluation batch size: 32 per device
- epochs: 3
- weight decay: 0.01
- regression loss: MSE
- primary model-selection metric: validation MAE

## Frozen routing experiment

The original frozen experiment selected threshold 0.94 using validation data only.

On the untouched test split:

- DistilBERT router mean performance: 0.760448
- GPT-4 fixed mean performance: 0.768067
- router historical mean cost: 0.002877
- GPT-4 historical mean cost: 0.003738
- observed performance gap: about 0.76 percentage points
- observed historical cost saving: about 23.05%

## Multi-seed robustness

Across three independent Kaggle training seeds (13, 42, 77), with the same
train/validation/test split:

- test router performance: 0.761428 ± 0.001359
- test router historical cost: 0.002966 ± 0.000093
- gap vs GPT-4: 0.664 ± 0.136 percentage points
- cost saving vs GPT-4: 20.65% ± 2.49%

These robustness runs are reported separately from the original frozen Colab run.

## Important limitations

- The candidate model pool and costs are historical RouterBench values.
- Free-form real-world prompts may be out of distribution.
- The model predicts routing scores; it does not generate answers.
- Paired-bootstrap uncertainty extended beyond the 1-percentage-point margin, so
  strict statistical non-inferiority at that margin was not established.
- A simple domain-aware baseline is very strong on this restricted three-domain subset.

## Attribution

RouterBench:
Qitian Jason Hu et al., ROUTERBENCH: A Benchmark for Multi-LLM Routing System,
arXiv:2403.12031 (2024).

DistilBERT:
Victor Sanh et al., DistilBERT, a distilled version of BERT: smaller, faster,
cheaper and lighter, arXiv:1910.01108 (2019).
"""

card_path = Path("/content/router_model_card.md")
card_path.write_text(MODEL_CARD, encoding="utf-8")

api.upload_file(
    path_or_fileobj=str(card_path),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add model card",
)

print("Model card uploaded.")

Model card uploaded.


In [7]:
files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model",
)

print("Repository files:")
for name in files:
    print(" -", name)

print("\nROUTER_MODEL_ID =", REPO_ID)

Repository files:
 - .gitattributes
 - README.md
 - config.json
 - model.safetensors
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin

ROUTER_MODEL_ID = Paam24/cost-aware-llm-router-distilbert


## Optional cleanup: remove training-only metadata from the Hub

`training_args.bin` is not required for inference. Run the next cell after publishing to keep the public model repository limited to the model/tokenizer/config artifacts.


In [ ]:
hub_files = api.list_repo_files(repo_id=REPO_ID, repo_type="model")

if "training_args.bin" in hub_files:
    api.delete_file(
        path_in_repo="training_args.bin",
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Remove inference-unneeded training_args.bin",
    )
    print("Deleted training_args.bin from Hugging Face Hub.")
else:
    print("training_args.bin is already absent.")

print(api.list_repo_files(repo_id=REPO_ID, repo_type="model"))
